In [ ]:
import pandas as pd
import os

INPUT_DATA_PATH = "../data/input"
OUTPUT_DATA_PATH = "../data/input"
NEW_DATA_FILE_NAME = "2000-2010_pop_data.xls"
OLD_DATA_FILE_NAME = "old_merged_data.csv"
OUTPUT_FILE_NAME = "old_merged_data.csv"

DROPPED_YEARS = [2000, 2001]
TARGET_YEARS = [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009]

In [ ]:
def load_excel_data(input_path: str, file_name: str) -> pd.DataFrame:
    data_path = os.path.join(input_path, file_name)
    return pd.read_excel(data_path)


def load_csv_data(input_path: str, file_name: str) -> pd.DataFrame:
    data_path = os.path.join(input_path, file_name)
    return pd.read_csv(data_path)


def correct_county_information(df: pd.DataFrame) -> pd.DataFrame:
    df["State"] = df["State"].str.replace(".", "")
    df["State"] = df["State"].str.replace(" County", ", GA")
    return df.rename(columns={"State": "County"})


def change_population_data_format(df: pd.DataFrame, dropped_years: list[int], target_years: list[int], id_columns: list[str]) -> pd.DataFrame:
    new_df = df.drop(dropped_years, axis=1)
    return pd.melt(
        frame=new_df,
        id_vars=id_columns,
        value_vars=target_years,
        var_name="Year",
        value_name="Population",
    ).sort_values(["County", "Year"])


def combine_df_population_data(old_df: pd.DataFrame, new_df: pd.DataFrame) -> pd.DataFrame:
    merged = old_df.merge(
        new_df[['County', 'Year', 'Population']],
        on=['County', 'Year'],
        how='left',
        suffixes=('', '_new')
    )

    merged['TOT_POP'] = merged['TOT_POP'].fillna(merged['Population']).astype(int)

    merged = merged.drop(columns=['Population'])
    return merged


def save_df_to_csv(df: pd.DataFrame, output_path: str, file_name: str) -> None:
    data_path = os.path.join(output_path, file_name)
    df.to_csv(data_path, index=False)

In [10]:
new_pop_df = load_excel_data(INPUT_DATA_PATH, NEW_DATA_FILE_NAME)
old_data_df = load_csv_data(INPUT_DATA_PATH, OLD_DATA_FILE_NAME)

correct_county_info_df = correct_county_information(new_pop_df)
reshaped_df = change_population_data_format(correct_county_info_df, DROPPED_YEARS, TARGET_YEARS, ["County"])

final_data_df = combine_df_population_data(old_data_df, reshaped_df)

save_df_to_csv(final_data_df, OUTPUT_DATA_PATH, OUTPUT_FILE_NAME)